In [1]:
import pandas as pd
train_df = pd.read_parquet("../Data/train/train.parquet")


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
train_df.head()

,level_2_id,gender,color,category_id,color_label,anonymous_id
0,932,NaN,Green,165,green,737d850c-c5f5-43bc-a4ef-b833cb0b866b
1,3356,NaN,Multicolore,54,multicolor,4b6913c9-46c6-4d6a-a1fd-19f9bf42e28d
2,435,NaN,White,75,white,5be2c43b-cbcc-40e7-a8b2-0620591ae628
3,6553,UNISEX,Black,152,black,ea01f5a8-3417-4de4-84d8-403870b318dd
4,443,NaN,Beige,82,beige,6c85a407-ef9a-42d3-ae4c-3c33b7851219


In [4]:
# 1. Shape
print(train_df.shape)

# 2. Null counts
print(train_df.isnull().sum())

# 3. Target distributions
print(train_df['category_id'].value_counts())
print(train_df['color_label'].value_counts())

# should print: 1    190  meaning every category_id maps to exactly one level_2_id

(31628, 6)
level_2_id          0
gender          25305
color            2582
category_id         0
color_label         0
anonymous_id        0
dtype: int64
category_id
68     188
81     188
33     187
138    187
28     187
      ... 
139    110
158     88
179     84
186     71
44       2
Name: count, Length: 191, dtype: int64
color_label
black         7022
white         3948
blue          2997
multicolor    2736
unknown       2725
gray          2527
brown         2253
red           2048
green         1740
pink          1721
yellow        1188
beige          723
Name: count, dtype: int64


In [5]:
print(train_df['gender'].value_counts())

# 4. Sanity check - doa category_id and level_2_id have a clean 1-to-1 mapping
print(train_df.groupby('category_id')['level_2_id'].nunique().value_counts())

gender
UNISEX    4021
FEMALE    1703
MALE       599
Name: count, dtype: int64
level_2_id
1    191
Name: count, dtype: int64


In [6]:
print(train_df['category_id'].value_counts()[train_df['category_id'].value_counts() < 50])

category_id
44    2
Name: count, dtype: int64


In [7]:
import os

image_dir = "../Data/train/images"  

def image_exists(row):
    path = os.path.join(image_dir, str(row['category_id']), f"{row['anonymous_id']}.jpg")
    return os.path.exists(path)

train_df['image_exists'] = train_df.apply(image_exists, axis=1)

print(train_df['image_exists'].value_counts())
print(f"Missing images: {(~train_df['image_exists']).sum()}")

image_exists
True    31628
Name: count, dtype: int64
Missing images: 0


In [8]:
import json

with open("../Data/level2_categories.json", "r") as f:
    mapping = json.load(f)

taxonomy = pd.DataFrame(mapping)

# Rename to match your dataframe
taxonomy = taxonomy.rename(columns={"new_id": "category_id", "google_id": "level_2_id"})

print(taxonomy.head())
print(taxonomy.shape)  # should be 192 rows

   category_id  level_2_id                                   category_name
0            1        3237           Animals & Pet Supplies > Live Animals
1            2           2           Animals & Pet Supplies > Pet Supplies
2            3        1604                Apparel & Accessories > Clothing
3            4         167    Apparel & Accessories > Clothing Accessories
4            5         184  Apparel & Accessories > Costumes & Accessories
(192, 3)


In [9]:
df = train_df.merge(taxonomy[['category_id', 'category_name']], on='category_id', how='left')

# Sanity check
print(df[['category_id', 'category_name']].drop_duplicates().sort_values('category_id'))
print(f"Any missing names: {df['category_name'].isna().sum()}")

     category_id                                   category_name
304            1           Animals & Pet Supplies > Live Animals
410            2           Animals & Pet Supplies > Pet Supplies
193            3                Apparel & Accessories > Clothing
318            4    Apparel & Accessories > Clothing Accessories
337            5  Apparel & Accessories > Costumes & Accessories
..           ...                                             ...
891          188           Toys & Games > Outdoor Play Equipment
256          189                          Toys & Games > Puzzles
49           190                             Toys & Games > Toys
177          191  Vehicles & Parts > Vehicle Parts & Accessories
213          192                     Vehicles & Parts > Vehicles

[191 rows x 2 columns]
Any missing names: 0


In [10]:
df

,level_2_id,gender,color,category_id,color_label,anonymous_id,image_exists,category_name
0,932,NaN,Green,165,green,737d850c-c5f5-43bc-a4ef-b833cb0b866b,True,Office Supplies > General Office Supplies
1,3356,NaN,Multicolore,54,multicolor,4b6913c9-46c6-4d6a-a1fd-19f9bf42e28d,True,Electronics > Arcade Equipment
2,435,NaN,White,75,white,5be2c43b-cbcc-40e7-a8b2-0620591ae628,True,"Food, Beverages & Tobacco > Tobacco Products"
3,6553,UNISEX,Black,152,black,ea01f5a8-3417-4de4-84d8-403870b318dd,True,Luggage & Bags > Train Cases
4,443,NaN,Beige,82,beige,6c85a407-ef9a-42d3-ae4c-3c33b7851219,True,Furniture > Chairs
...,...,...,...,...,...,...,...,...
31623,2636,NaN,Nero,175,black,7dadb4da-8999-49ff-89b4-01e1c56aeabd,True,Office Supplies > Shipping Supplies
31624,499982,NaN,Red,112,red,98d3f9fa-cecd-4414-9470-8aeddcd165c8,True,Hardware > Small Engines
31625,3650,NaN,White,114,white,1f19012e-89cb-4999-977c-780771e363e0,True,Hardware > Tool Accessories
31626,7261,NaN,Grau,27,gray,3e517d3e-4dd1-4ebb-b0e3-3e9c461afd1b,True,Business & Industrial > Automation Control Com...


In [11]:
#split category_name into category and subcategory
df[['category', 'subcategory']] = df['category_name'].str.split(' > ', expand=True)

In [12]:
# Check what class 44 is
print(df[df['category_id'] == 44]['category_name'].values)

# Find a similar class to merge into - probably Construction (28) or Heavy Machinery (35)
print(df[df['category_id'] == 28]['category_name'].values)

# Reassign class 44 samples to class 28
df.loc[df['category_id'] == 44, 'category_id'] = 28

<ArrowStringArray>
['Business & Industrial > Mining & Quarrying', 'Business & Industrial > Mining & Quarrying']
Length: 2, dtype: str
<ArrowStringArray>
['Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 ...
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction',
 'Business & Industrial > Construction']
Length: 187, dtype: st

In [13]:
from sklearn.preprocessing import LabelEncoder
import pickle

# Encode color
color_encoder = LabelEncoder()
df['color_encoded'] = color_encoder.fit_transform(df['color_label'])

# Encode gender - fill nulls first
df['gender_filled'] = df['gender'].fillna('UNKNOWN')
gender_encoder = LabelEncoder()
df['gender_encoded'] = gender_encoder.fit_transform(df['gender_filled'])

# Encode category
cat_encoder = LabelEncoder()
df['category_encoded'] = cat_encoder.fit_transform(df['category_id'])

# Check
print(dict(zip(color_encoder.classes_, color_encoder.transform(color_encoder.classes_))))
print(dict(zip(gender_encoder.classes_, gender_encoder.transform(gender_encoder.classes_))))
print(f"Min category encoded: {df['category_encoded'].min()}")
print(f"Max category encoded: {df['category_encoded'].max()}")
print(f"Unique categories: {df['category_encoded'].nunique()}")

# Save encoders
with open("color_encoder.pkl", "wb") as f:
    pickle.dump(color_encoder, f)

with open("gender_encoder.pkl", "wb") as f:
    pickle.dump(gender_encoder, f)

with open("category_encoder.pkl", "wb") as f:
    pickle.dump(cat_encoder, f)

{'beige': np.int64(0), 'black': np.int64(1), 'blue': np.int64(2), 'brown': np.int64(3), 'gray': np.int64(4), 'green': np.int64(5), 'multicolor': np.int64(6), 'pink': np.int64(7), 'red': np.int64(8), 'unknown': np.int64(9), 'white': np.int64(10), 'yellow': np.int64(11)}
{'FEMALE': np.int64(0), 'MALE': np.int64(1), 'UNISEX': np.int64(2), 'UNKNOWN': np.int64(3)}
Min category encoded: 0
Max category encoded: 189
Unique categories: 190


In [14]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['category_id'], random_state=42)

print(f"Train size: {len(train_df)} | Classes: {train_df['category_id'].nunique()}")
print(f"Test size:  {len(test_df)}  | Classes: {test_df['category_id'].nunique()}")

Train size: 25302 | Classes: 190
Test size:  6326  | Classes: 190


In [15]:
print(f"Train size: {len(train_df)}")
print(f"Test size:  {len(test_df)}")

# Make sure all 190 classes are in the train set
print(f"Classes in train: {train_df['category_id'].nunique()}")
print(f"Classes in test:  {test_df['category_id'].nunique()}")

Train size: 25302
Test size:  6326
Classes in train: 190
Classes in test:  190


In [16]:
train_df.head()

,level_2_id,gender,color,category_id,color_label,anonymous_id,image_exists,category_name,category,subcategory,color_encoded,gender_filled,gender_encoded,category_encoded
19988,976,NaN,Multicolor,48,multicolor,0a4f57c5-30f6-48e9-9589-acf30d6c9fd4,True,Business & Industrial > Signage,Business & Industrial,Signage,6,UNKNOWN,3,46
23057,887,FEMALE,Black,161,black,5aa3cb76-ebb1-4731-9ef1-d2c8cb87e44c,True,Media > Sheet Music,Media,Sheet Music,1,FEMALE,0,158
9949,5606,NaN,green,176,green,856849e7-ba89-487c-af29-448b9e9cdcb1,True,Religious & Ceremonial > Memorial Ceremony Sup...,Religious & Ceremonial,Memorial Ceremony Supplies,5,UNKNOWN,3,173
16764,2496,UNISEX,Blue,43,blue,616868dc-3978-4f8c-87cf-346540192b53,True,Business & Industrial > Medical,Business & Industrial,Medical,2,UNISEX,2,42
19348,114,NaN,blanc,28,white,9b3e5192-5649-4434-9b64-e29b94122059,True,Business & Industrial > Construction,Business & Industrial,Construction,10,UNKNOWN,3,27


In [17]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class ProductDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = os.path.join(self.image_dir, str(row['category_id']), f"{row['anonymous_id']}.jpg")
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        category = int(row['category_encoded'])  
        color    = int(row['color_encoded'])
        gender   = int(row['gender_encoded'])
        
        return image, category, color, gender

image_dir = r"C:\Users\karim\Desktop\Data Science Projects\Vast AI\Data\train\images"

train_dataset = ProductDataset(train_df, image_dir, transform=train_transforms)
test_dataset  = ProductDataset(test_df,  image_dir, transform=test_transforms)

from torch.utils.data import WeightedRandomSampler

class_counts   = train_df['category_encoded'].value_counts().sort_index()
weights        = 1.0 / class_counts
sample_weights = train_df['category_encoded'].map(weights).values
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,  num_workers=0)

images, categories, colors, genders = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")
print(f"Category labels:   {categories.shape}")
print(f"Color labels:      {colors.shape}")
print(f"Gender labels:     {genders.shape}")
print(f"Category min: {categories.min()} | max: {categories.max()}")  # should be 0-190

Image batch shape: torch.Size([32, 3, 224, 224])
Category labels:   torch.Size([32])
Color labels:      torch.Size([32])
Gender labels:     torch.Size([32])
Category min: 0 | max: 186


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = int(train_df["category_encoded"].nunique())

efficientnet = models.efficientnet_b3(weights="IMAGENET1K_V1")
efficientnet.classifier[1] = nn.Linear(efficientnet.classifier[1].in_features, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = efficientnet.to(device)
print(f"Device: {device}")
print(f"Classes: {NUM_CLASSES}")

Device: cuda
Classes: 190


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for param in model.features.parameters():
    param.requires_grad = False

optimizer = Adam(model.classifier.parameters(), lr=1e-3)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct = 0, 0

    for images, categories, colors, genders in loader:
        images     = images.to(device)
        categories = categories.to(device)

        optimizer.zero_grad()
        cat_out = model(images)
        loss    = criterion(cat_out, categories)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (cat_out.argmax(1) == categories).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

# Phase 1 - 5 epochs
print("Phase 1 - classifier only")
for epoch in range(5):
    loss, acc = train_epoch(model, train_loader, optimizer)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.4f}")

# Phase 2 - unfreeze everything
print("Phase 2 - full fine-tuning")
for param in model.features.parameters():
    param.requires_grad = True

optimizer = Adam([
    {'params': model.features.parameters(),   'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-4, 'weight_decay': 1e-4},
])

scheduler = CosineAnnealingLR(optimizer, T_max=20)

best_acc  = 0
ckpt_path = "efficientnet_b3_best.pth"

for epoch in range(20):
    loss, acc = train_epoch(model, train_loader, optimizer)
    scheduler.step()
    print(f"Epoch {epoch+1}/20 | Loss: {loss:.4f} | Acc: {acc:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), ckpt_path)
        print(f"  → Saved new best model ({best_acc:.4f})")

Phase 1 - classifier only
Epoch 1 | Loss: 4.0977 | Acc: 0.1926
Epoch 2 | Loss: 3.5423 | Acc: 0.2692
Epoch 3 | Loss: 3.3588 | Acc: 0.2963
Epoch 4 | Loss: 3.2468 | Acc: 0.3130
Epoch 5 | Loss: 3.1696 | Acc: 0.3246
Phase 2 - full fine-tuning
Epoch 1 | Loss: 2.8709 | Acc: 0.3753
  → Saved new best model (0.3753)
Epoch 2 | Loss: 2.7082 | Acc: 0.4065
  → Saved new best model (0.4065)
Epoch 3 | Loss: 2.6035 | Acc: 0.4218
  → Saved new best model (0.4218)
Epoch 4 | Loss: 2.5273 | Acc: 0.4359
  → Saved new best model (0.4359)
Epoch 5 | Loss: 2.4487 | Acc: 0.4525
  → Saved new best model (0.4525)
Epoch 6 | Loss: 2.4014 | Acc: 0.4613
  → Saved new best model (0.4613)
Epoch 7 | Loss: 2.3153 | Acc: 0.4757
  → Saved new best model (0.4757)
Epoch 8 | Loss: 2.2690 | Acc: 0.4853
  → Saved new best model (0.4853)
Epoch 9 | Loss: 2.2305 | Acc: 0.4908
  → Saved new best model (0.4908)
Epoch 10 | Loss: 2.1651 | Acc: 0.5017
  → Saved new best model (0.5017)


In [30]:
from sklearn.metrics import f1_score

model.load_state_dict(torch.load("./efficientnet_best.pth"))
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for images, categories, colors, genders in test_loader:
        images     = images.to(device)
        categories = categories.to(device)
        cat_out    = model(images)
        
        all_preds.extend(cat_out.argmax(1).cpu().numpy())
        all_labels.extend(categories.cpu().numpy())

# Overall accuracy
correct = sum(p == l for p, l in zip(all_preds, all_labels))
print(f"Test Accuracy:   {correct/len(all_labels):.4f}")
print(f"Overfitting gap: {best_acc - correct/len(all_labels):.4f}")

# F1 scores
print(f"F1 Macro:        {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"F1 Weighted:     {f1_score(all_labels, all_preds, average='weighted'):.4f}")

C:\Users\karim\AppData\Local\Temp\ipykernel_13960\3984954762.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("./efficientnet_best.pth"))

<All keys matched successfully>

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

Test Accuracy:   0.4240
Overfitting gap: 0.0778
F1 Macro:        0.4044
F1 Weighted:     0.4073
